In [1]:
# Part 1 - Create the Project List (Test Type Detection Only)
#========= create Project_List basic ====================

import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Project_List.csv")

# === CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'Firebase_Full': ['gcloud firebase test android run'],
    'Firebase_Compact': ['Firebase-Test-Lab-Action'],
    'Appcenter': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'Browserstack': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'emulator_manual': ['create avd'],
    'GitHub_GMD': ['cleanManagedDevices'],
    'Unit_Test': ['gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
                  'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'],
    'Other': ['instrumentation', 'instrument']
}

# === STRUCTURE TO HOLD RESULTS ===
project_results = {}

# === DETECTION LOGIC ===
def detect_testing_types(yaml_text):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found

# === MAIN PARSER ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            detected = detect_testing_types(raw)
            content = yaml.safe_load(raw)
            if not content:
                return {'types': detected, 'error': True}
            return {'types': detected, 'error': False}
    except Exception:
        return {'types': set(), 'error': True}

# === PROJECT SCANNER ===
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

            result = parse_yaml_file(file_path)

            if project_name not in project_results:
                project_results[project_name] = {'types': set(), 'errors': 0}
            project_results[project_name]['types'].update(result['types'])
            if result['error']:
                project_results[project_name]['errors'] += 1

# === EXPORT CSV ===
rows = []
for project, result in project_results.items():
    cleaned_types = result['types']
    if 'Other' in cleaned_types and len(cleaned_types) > 1:
        cleaned_types = cleaned_types - {'Other'}

    rows.append({
        'project': project,
        'test_types': ', '.join(sorted(cleaned_types)) if cleaned_types else 'none',
        'yaml_errors': result['errors']
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
Project_List = df.copy()

print(f"\n✅ Summary written to: {OUTPUT_CSV}")



✅ Summary written to: C:\GitHub\Android-Mobile-Apps\Project_List.csv


In [2]:
#========= create YML_List ====================
import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "YML_List.csv")

# === CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'Firebase_Full': ['gcloud firebase test android run'],
    'Firebase_Compact': ['Firebase-Test-Lab-Action'],
    'Appcenter': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'Browserstack': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'GitHub_GMD': ['cleanManagedDevices']
    #'Other': ['instrumentation', 'instrument']
}

UNIT_TEST_KEYWORDS = [
    'gradlew test', './gradlew test', 'testdebugunittest', 'testreleaseunittest',
    'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

MANUAL_EMULATOR_KEYWORDS = ['create avd']

# === CI PLATFORM DETECTION ===
def detect_ci_platform(file_path, yaml_text):
    text = yaml_text.lower()
    filename = os.path.basename(file_path).lower()
    filepath = file_path.lower()

    if ".gitlab-ci.yml" in filename or "gitlab" in filepath:
        return "GitLab CI"
    elif ".travis.yml" in filename or "travis" in filepath:
        return "Travis CI"
    elif "circleci" in filepath or "circleci" in filename:
        return "CircleCI"
    elif "bitrise" in filepath or "bitrise" in filename:
        return "Bitrise"
    elif ".github" in filepath or "/.github/" in filepath:
        return "GitHub"
    else:
        return "Unknown"


# === DETECT OTHER TEST TYPES ===
def detect_test_type(yaml_text):
    text = yaml_text.lower()
    detected = []
    for test_type, keywords in TEST_TYPES.items():
        for keyword in keywords:
            if keyword.lower() in text:
                detected.append(test_type)
                break
    return ', '.join(detected) if detected else 'None'

# === DETECT UNIT TEST ===
def detect_unit_test(yaml_text):
    text = yaml_text.lower()
    return any(keyword in text for keyword in UNIT_TEST_KEYWORDS)

# === DETECT MANUAL EMULATOR BY CI ===
def detect_manual_emulator(text, ci_platform):
    for keyword in MANUAL_EMULATOR_KEYWORDS:
        if keyword in text:
            return f"{ci_platform}_emulator_manual" if ci_platform != "Unknown" else "Other_emulator_manual"
    return None

# === PROCESS YAML FILES ===
project_results = []
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path, raw)
                    unit_test = detect_unit_test(raw)
                    test_type = detect_test_type(raw)
                    manual_emulator = detect_manual_emulator(raw.lower(), ci_platform)

                    # Append manual emulator to test_type if found
                    if manual_emulator:
                        test_type = f"{test_type}, {manual_emulator}" if test_type != 'None' else manual_emulator

                    # Determine if instrumentation test is present
                    instrumentation_test = False if test_type.strip().lower() in ['none', 'other'] else True

                    filename = os.path.basename(file_path)
                    parts = filename.split(".")
                    project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

                    project_results.append({
                        'project': project_name,
                        'ci_platform': ci_platform,
                        'test_type': test_type,
                        'unit_test': unit_test,
                        'instrumentation_test': instrumentation_test
                    })
            except Exception:
                project_results.append({
                    'project': project_name,
                    'ci_platform': 'Error',
                    'test_type': 'Error',
                    'unit_test': False,
                    'instrumentation_test': False
                })

# === EXPORT TO CSV ===
df = pd.DataFrame(project_results)
df.to_csv(OUTPUT_CSV, index=False)
YML_List = df.copy()
print(f"\n✅ YML summary written to: {OUTPUT_CSV}")



✅ YML summary written to: C:\GitHub\Android-Mobile-Apps\YML_List.csv
